In [ ]:

import sys
from pathlib import Path
sys.path.append(str(Path("..").resolve() / "src"))
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans

analysis_df = pd.read_parquet("weedvlm_scored_responses.parquet")

print(analysis_df.columns)
print(analysis_df["task"].unique())

Index(['id', 'run', 'model', 'question_id', 'status', 'answer', 'reasoning',
       'parsed', 'raw_response', 'warnings', 'error_kind', 'error_message',
       'error_http_status', 'attempts', 'retry_log', 'finish_reason',
       'latency_seconds', 'usage', 'total_tokens', 'cost_usd', 'generation_id',
       'provider', 'image_sha256', 'prompt_version', 'created_at', 'task',
       'position', 'question_type', 'question_text', 'image_id', 'annotated',
       'benchmark_class', 'dataset_name', 'source_image_id', 'annotation_ids',
       'expected_answer', 'choices', 'answer_index', 'ablation_of',
       'true_weed_count', 'true_weed_coverage_fraction', 'target_species',
       'image_width', 'image_height', 'boxes', 'match', 'outcome',
       'localisation_score', 'matched_iou', 'est_count', 'est_coverage_pct',
       'count_error', 'abs_count_error', 'coverage_error_pp',
       'abs_coverage_error_pp'],
      dtype='str')
<ArrowStringArray>
['species_id', 'fine_grained', 'density', 'gr

In [13]:
def cluster_task(df: pd.DataFrame, n_clusters=6):
    data = df.dropna(subset=["reasoning"]).copy()

    vectorizer = TfidfVectorizer(
        lowercase=True,
        stop_words="english",
        ngram_range=(1,2),
        min_df=5,
        max_df=0.90,
        sublinear_tf=True,
    )

    X = vectorizer.fit_transform(data["reasoning"])
    model = KMeans(
        n_clusters=n_clusters,
        random_state=42,
        n_init="auto",
    )
    data["cluster"] = model.fit_predict(X)

    return data, vectorizer, model

results = []
cluster_models = {}

for task, task_df in analysis_df.groupby("task"):
    clustered, vectorizer, cluster_model = cluster_task(task_df)
    results.append(clustered)
    cluster_models[task] = {
        "vectorizer": vectorizer,
        "model": cluster_model,
    }

analysis_df = pd.concat(results, ignore_index=True)
print(analysis_df.head())

     id                    run                       model  \
0  1834  claude-haiku-full-run  anthropic/claude-haiku-5.5   
1  1837  claude-haiku-full-run  anthropic/claude-haiku-5.5   
2  1839  claude-haiku-full-run  anthropic/claude-haiku-5.5   
3  1840  claude-haiku-full-run  anthropic/claude-haiku-5.5   
4  1841  claude-haiku-full-run  anthropic/claude-haiku-5.5   

                                         question_id status answer  \
0           CropAndWeed:274:density_estimation:boxed     ok    Low   
1  MultipleWeedSpeciesDetection:62:density_estima...     ok    Low   
2      CottonWeedDet12:2901:density_estimation:boxed     ok    Low   
3      CottonWeedDet12:3362:density_estimation:boxed     ok    Low   
4      CottonWeedDet12:1286:density_estimation:boxed     ok    Low   

                                           reasoning  \
0  Three bounding-boxed weeds are present: a smal...   
1  There is a single weed plant outlined by the b...   
2  The image contains a single weed pl

In [14]:
fine_grained_by_class = (
    analysis_df[analysis_df["task"] == "fine_grained"]
    .groupby(["cluster", "benchmark_class"])
    .agg(
        n=("match", "size"),
        accuracy=("match", "mean"),
    )
    .reset_index()
)
print(fine_grained_by_class)

    cluster    benchmark_class    n  accuracy
0         0  different_species  232  0.793103
1         0       same_species  198  0.929293
2         1  different_species  122  0.868852
3         1       same_species   34  0.235294
4         2  different_species  257  0.992218
5         2       same_species   51  0.039216
6         3  different_species   47       0.0
7         3       same_species  186       1.0
8         4  different_species   47  0.106383
9         4       same_species  356  0.977528
10        5  different_species  195  0.369231
11        5       same_species   75  0.133333


In [11]:
fg = analysis_df[
    analysis_df["task"] == "fine_grained"
].copy()

for cluster in sorted(fg["cluster"].unique()):
    print(f"\n{'=' * 80}")
    print(f"CLUSTER {cluster}")
    print("=" * 80)

    sample = (
        fg[fg["cluster"] == cluster]
        [["benchmark_class", "answer", "match", "reasoning"]]
        .sample(
            min(10, len(fg[fg["cluster"] == cluster])),
            random_state=42
        )
    )

    for _, row in sample.iterrows():
        print(
            f"\nTruth: {row['benchmark_class']}"
            f" | Answer: {row['answer']}"
            f" | Correct: {row['match']}"
        )
        print(row["reasoning"])


CLUSTER 0

Truth: different_species | Answer: Different species | Correct: True
Box 1 contains a sprawling plant with reddish stems and small, smooth, fleshy leaves. The plant in box 2 has broader leaves with toothed or lobed edges, unlike the plant in box 1.

Truth: different_species | Answer: Different species | Correct: True
Box 1 has small, rounded, spoon-shaped leaves on a reddish stem, which resembles purslane seedlings. Box 2 has longer, lanceolate leaves with pronounced veins and a thicker stem, which looks like a different broadleaf weed.

Truth: same_species | Answer: Same species | Correct: True
Both plants share identical morphological traits, such as fleshy, spatulate leaves and reddish stems, indicating they belong to the same species. The differences in size and leaf damage are due to growth stage and environmental factors.

Truth: same_species | Answer: Same species | Correct: True
Both plants have thick, fleshy, spoon-shaped leaves clustered around reddish, succulent 

In [16]:
vectorizer = cluster_models["fine_grained"]["vectorizer"]
clusterer = cluster_models["fine_grained"]["model"]

terms = vectorizer.get_feature_names_out()

n_terms = 15

for cluster_id, centroid in enumerate(clusterer.cluster_centers_):
    top_indices = centroid.argsort()[::-1][:n_terms]

    print(f"\nCluster {cluster_id}:")
    print(", ".join(terms[top_indices]))


Cluster 0:
plant, box, leaves, reddish, stems, plant box, prostrate, small, opposite, species, purslane, reddish stems, fleshy, rounded, leaf

Cluster 1:
box contains, contains, bounding box, box, bounding, contains small, plant, small, grass, broadleaf, different, leaves, contains grass, contains plant, species

Cluster 2:
plant, leaves, different, leaves plant, shapes, leaf shapes, narrow, forms, different species, indicate, leaf, broad, plant box, box, long

Cluster 3:
exhibit, plants exhibit, characteristic, growth habit, habit, plants, growth, identical, morphology, leaf, leaf morphology, exhibit identical, species, bounding boxes, identical leaf

Cluster 4:
similar, plants, consistent, leaves, growth, smooth, size, leaf, leaves similar, plants similar, differences, stems, stage, box, growth stage

Cluster 5:
box, shows, box shows, plant, seedling, tiny, small, leaves, plant box, reliably, species, confirm, features, box tiny, lobed


In [17]:
def cluster_performace(df: pd.DataFrame):
    results = []
    # Categorical tasks, non-separate
    categorical_tasks = [
        "fine_grained",
        "grounded_vqa",
    ]

    for task in categorical_tasks:
        x = (
            df[df["task"] == task]
            .groupby("cluster")
            .agg(
                n=("match", "size"),
                accuracy=("match", "mean"),
            )
            .reset_index()
        )
        x["task"] = task
        results.append(x)
    species_cluster_performace = (
        df[df["task"] == "species_id"]
        .groupby(["cluster", "question_type"])
        .agg(
            n=("match", "size"),
            accuracy=("match", "mean"),
        )
        .reset_index()
    )
    results.append(species_cluster_performace)

In [19]:
species_cluster_performace = (
    analysis_df[analysis_df["task"] == "species_id"]
    .groupby(["cluster", "question_type"])
    .agg(
        n=("match", "size"),
        accuracy=("match", "mean"),
    )
    .reset_index()
)
print(species_cluster_performace)

    cluster               question_type     n  accuracy
0         0  species_id_multiple_choice   652  0.912577
1         0       species_id_open_ended   109  0.623853
2         1  species_id_multiple_choice   849  0.771496
3         1       species_id_open_ended   160    0.1875
4         2  species_id_multiple_choice  2058   0.82896
5         2       species_id_open_ended   563  0.127886
6         3  species_id_multiple_choice   324  0.962963
7         3       species_id_open_ended    25      0.84
8         4  species_id_multiple_choice  1154  0.904679
9         4       species_id_open_ended   124  0.290323
10        5  species_id_multiple_choice   395  0.979747
11        5       species_id_open_ended   104  0.894231


In [20]:
sid = analysis_df[
    analysis_df["task"] == "species_id"
].copy()

for cluster in sorted(sid["cluster"].unique()):
    print("\n" + "=" * 80)
    print(f"CLUSTER {cluster}")
    print("=" * 80)

    sample = (
        sid[sid["cluster"] == cluster]
        [
            [
                "question_type",
                "benchmark_class",
                "answer",
                "expected_answer",
                "match",
                "reasoning",
            ]
        ]
        .sample(
            min(10, len(sid[sid["cluster"] == cluster])),
            random_state=42,
        )
    )

    for _, row in sample.iterrows():
        print(
            f"\nType: {row['question_type']}"
            f" | Correct: {row['match']}"
        )
        print(f"Expected: {row['expected_answer']}")
        print(f"Answer: {row['answer']}")
        print(row["reasoning"])


CLUSTER 0

Type: species_id_multiple_choice | Correct: True
Expected: White goosefoot
Answer: White goosefoot
The plant exhibits the characteristic rhomboid or triangular leaf shape and mealy texture typical of Chenopodium album. The overall growth habit and leaf morphology are consistent with White goosefoot.

Type: species_id_multiple_choice | Correct: True
Expected: Xanthium strumarium
Answer: Xanthium strumarium
The seedling has broad, triangular, irregularly lobed and toothed leaves with a rough appearance, characteristic of common cocklebur (Xanthium strumarium). The leaf shape and opposite arrangement distinguish it from Chenopodium album and Amaranthus tuberculatus.

Type: species_id_multiple_choice | Correct: True
Expected: White goosefoot
Answer: White goosefoot
The plant has branched stems with diamond-shaped, toothed, alternate leaves, consistent with Chenopodium album. The leaf shape rules out the finely divided leaves of common fumitory and the narrow opposite leaves of 

In [21]:
sid = analysis_df[
    analysis_df["task"] == "species_id"
].copy()

species_cluster = pd.crosstab(
    sid["benchmark_class"],
    sid["cluster"],
    normalize="index",
)

display(species_cluster.round(2))

cluster,0,1,2,3,4,5
benchmark_class,,,,,,
Amaranthus tuberculatus,0.01,0.19,0.76,0.02,0.00,0.00
Ambrosia artemisiifolia,0.00,0.22,0.06,0.00,0.72,0.00
Chenopodium album,0.44,0.17,0.29,0.00,0.09,0.00
Cockspur grass,0.00,0.08,0.91,0.00,0.01,0.00
Creeping thistle,0.00,0.20,0.10,0.00,0.70,0.00
Eclipta,0.00,0.19,0.80,0.00,0.00,0.01
Euphorbia maculata,0.00,0.17,0.79,0.00,0.00,0.03
German chamomile,0.00,0.18,0.07,0.00,0.76,0.00
Hybrid goosefoot,0.74,0.09,0.11,0.00,0.06,0.00


In [22]:
cluster_species = pd.crosstab(
    sid["cluster"],
    sid["benchmark_class"],
    normalize="index",
)

display(cluster_species.round(2))

benchmark_class,Amaranthus tuberculatus,Ambrosia artemisiifolia,Chenopodium album,Cockspur grass,Creeping thistle,Eclipta,Euphorbia maculata,German chamomile,Hybrid goosefoot,Ipomoea indica,Mollugo verticillata,Physalis angulata,Portulaca oleracea,Red root amaranth,Senna obtusifolia,Sida rhombifolia,Taraxacum officinale,Thorn apple,White goosefoot,Xanthium strumarium
cluster,,,,,,,,,,,,,,,,,,,,
0,0.00,0.00,0.11,0.00,0.00,0.00,0.00,0.00,0.22,0.00,0.00,0.01,0.00,0.01,0.00,0.00,0.00,0.01,0.53,0.10
1,0.04,0.09,0.03,0.02,0.04,0.08,0.03,0.04,0.02,0.10,0.04,0.05,0.00,0.09,0.01,0.09,0.09,0.04,0.01,0.07
2,0.06,0.01,0.02,0.07,0.01,0.13,0.06,0.01,0.01,0.13,0.06,0.06,0.00,0.12,0.02,0.14,0.02,0.01,0.01,0.04
3,0.01,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.01,0.00,0.00,0.00,0.00,0.97,0.00,0.00,0.00,0.00,0.00
4,0.00,0.25,0.01,0.00,0.11,0.00,0.00,0.13,0.01,0.00,0.00,0.01,0.00,0.01,0.00,0.00,0.22,0.12,0.01,0.13
5,0.00,0.00,0.00,0.00,0.00,0.01,0.01,0.00,0.00,0.00,0.01,0.00,0.96,0.00,0.00,0.00,0.00,0.00,0.00,0.00
